# Ingest Selected Session IDs Into One MLflow Run

填入 `SESSION_IDS` 後執行，即可把多個 `session_id` 的 traces 匯入同一個 MLflow run。

In [1]:
import json
import os
import uuid
from datetime import datetime

import mlflow
from dotenv import load_dotenv
from mlflow.environment_variables import MLFLOW_EXPERIMENT_ID

from codex_session_ingest import ingest_session_by_id

/Users/bocheng/Side-project/codex-test/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

# ===== 使用者設定區 =====
DOTENV_PATH = ".env"
EXPERIMENT_ID = os.getenv("MLFLOW_EXPERIMENT_ID", "").strip()
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "").strip()
RUN_NAME_BASE = "ingest-selected-sessions"
KEEP_EXISTING_SESSION_TRACES = False

# 直接填你要匯入的 session_id 清單
SESSION_IDS = [
    "019cf226-f3e3-7cf3-a8c2-bedb0f59dd2f",
    "019cf223-fcec-7b43-ad58-5490cf266f06"
]

if not SESSION_IDS:
    raise ValueError("請先在 SESSION_IDS 填入至少一個 session_id")

In [3]:
load_dotenv(DOTENV_PATH)

if EXPERIMENT_ID:
    exp = mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
elif EXPERIMENT_NAME:
    exp = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
else:
    # fallback 到 .env 的設定
    env_exp_id = os.getenv("MLFLOW_EXPERIMENT_ID", "").strip()
    env_exp_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "").strip()
    if env_exp_id:
        exp = mlflow.set_experiment(experiment_id=env_exp_id)
    elif env_exp_name:
        exp = mlflow.set_experiment(experiment_name=env_exp_name)
    else:
        raise RuntimeError("請提供 EXPERIMENT_ID/EXPERIMENT_NAME，或在 .env 設定 MLFLOW_EXPERIMENT_ID/MLFLOW_EXPERIMENT_NAME")

# 讓 codex_session_ingest traces 寫到同一個 experiment
os.environ[MLFLOW_EXPERIMENT_ID.name] = exp.experiment_id

run_name = f"{RUN_NAME_BASE}-{datetime.now().strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:8]}"
results = []

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("session_id_count", len(SESSION_IDS))
    mlflow.log_param("keep_existing_session_traces", KEEP_EXISTING_SESSION_TRACES)

    for sid in SESSION_IDS:
        item = ingest_session_by_id(
            session_id=sid,
            delete_existing_session=not KEEP_EXISTING_SESSION_TRACES,
        )
        results.append(item)

    total_traces = sum(len(r.get("traces", [])) for r in results)
    total_turns = sum(int(r.get("turn_count", 0)) for r in results)
    total_deleted = sum(int(r.get("deleted_trace_count", 0)) for r in results)

    mlflow.log_metric("ingested_session_count", len(results))
    mlflow.log_metric("ingested_trace_count", total_traces)
    mlflow.log_metric("ingested_turn_count", total_turns)
    mlflow.log_metric("deleted_trace_count", total_deleted)

    out = {
        "run_id": run.info.run_id,
        "experiment_id": run.info.experiment_id,
        "run_name": run_name,
        "session_ids": SESSION_IDS,
        "results": results,
    }

    mlflow.log_dict(out, "selected_session_ingest_results.json")

out

/Users/bocheng/Side-project/codex-test/codex_session_ingest/ingest.py:451: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  page = client.search_traces(
/Users/bocheng/Side-project/codex-test/codex_session_ingest/ingest.py:451: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  page = client.search_traces(


🏃 View run ingest-selected-sessions-20260316-004058-d356292e at: https://dbc-4a3d787b-adc8.cloud.databricks.com/ml/experiments/4439623448865416/runs/bf02a5db6a4641ae940dd19e5fab01e8
🧪 View experiment at: https://dbc-4a3d787b-adc8.cloud.databricks.com/ml/experiments/4439623448865416


{'run_id': 'bf02a5db6a4641ae940dd19e5fab01e8',
 'experiment_id': '4439623448865416',
 'run_name': 'ingest-selected-sessions-20260316-004058-d356292e',
 'session_ids': ['019cf226-f3e3-7cf3-a8c2-bedb0f59dd2f',
  '019cf223-fcec-7b43-ad58-5490cf266f06'],
 'results': [{'session_id': '019cf226-f3e3-7cf3-a8c2-bedb0f59dd2f',
   'turn_count': 3,
   'event_count': 33,
   'usage': {'input_tokens': 113424,
    'cached_input_tokens': 94976,
    'output_tokens': 458,
    'total_tokens': 113882,
    'token_count': 208858},
   'deleted_trace_count': 3,
   'session_path': '/Users/bocheng/.codex/sessions/2026/03/15/rollout-2026-03-15T23-39-25-019cf226-f3e3-7cf3-a8c2-bedb0f59dd2f.jsonl',
   'traces': [{'trace_id': 'tr-5810dd5d79a7b55965e67742140cf6d2',
     'session_id': '019cf226-f3e3-7cf3-a8c2-bedb0f59dd2f',
     'turn_id': '019cf227-2800-77f2-baa2-802a8abf70f8',
     'turn_index': 0,
     'request': 'hi 我現在有哪些 mcp server',
     'response_preview': '**MCP Servers**\n\n- 当前没有可访问的 MCP 服务器资源（`list_mcp_res